In [1]:
!git clone https://github.com/Zhuzhik365/chat_bot_3adapters_dynamic_hybrid
%cd chat_bot_3adapters_dynamic_hybrid


Cloning into 'ai-consultant-chatbot'...
remote: Enumerating objects: 43, done.
remote: Counting objects: 100% (1/1), done.
remote: Total 43 (delta 0), reused 0 (delta 0), pack-reused 42 (from 2)
Receiving objects: 100% (43/43), 75.14 MiB | 44.04 MiB/s, done.
Resolving deltas: 100% (3/3), done.


In [2]:
!pip install -q -r requirements.txt


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.2 MB/s eta 0:00:00:00:0100:01


In [3]:
!pip install --upgrade transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 90.9 MB/s eta 0:00:00:00:01:01
  Attempting uninstall: transformers
    Found existing installation: transformers 5.2.0
    Uninstalling transformers-5.2.0:
      Successfully uninstalled transformers-5.2.0


In [4]:
import gc
import os
import threading
import torch
from dotenv import load_dotenv
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from flask import Flask, request, jsonify
from pyngrok import ngrok


def get_my_secret(secret_name, default=""):
    try:
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret(secret_name)
        if value:
            return value
    except Exception:
        pass

    try:
        from google.colab import userdata
        value = userdata.get(secret_name)
        if value:
            return value
    except Exception:
        pass

    try:
        load_dotenv('.env')
        value = os.getenv(secret_name)
        if value:
            return value
    except Exception:
        pass

    return os.environ.get(secret_name, default)


In [5]:
BASE_MODEL_NAME = "Qwen/Qwen3.5-4B"
REPO_NAME = "chat_bot_3adapters_dynamic_hybrid"

ADAPTER_PATH_CANDIDATES = {
    "business": [
        f"/kaggle/working/{REPO_NAME}/adapters/business_adapter",
        f"/content/{REPO_NAME}/adapters/business_adapter",
        "/content/drive/MyDrive/adapters/business_adapter",
        "/content/adapters/business_adapter",
        "./adapters/business_adapter",
        "adapters/business_adapter",
        "/kaggle/working/ai-consultant-chatbot/adapters/business_adapter",
        "/content/ai-consultant-chatbot/adapters/business_adapter",
    ],
    "legal": [
        f"/kaggle/working/{REPO_NAME}/adapters/laws_adapter",
        f"/content/{REPO_NAME}/adapters/laws_adapter",
        "/content/drive/MyDrive/adapters/laws_adapter",
        "/content/adapters/laws_adapter",
        "./adapters/laws_adapter",
        "adapters/laws_adapter",
        "/kaggle/working/ai-consultant-chatbot/adapters/laws_adapter",
        "/content/ai-consultant-chatbot/adapters/laws_adapter",
    ],
    "psych": [
        f"/kaggle/working/{REPO_NAME}/adapters/psych_adapter",
        f"/content/{REPO_NAME}/adapters/psych_adapter",
        "/content/drive/MyDrive/adapters/psych_adapter",
        "/content/adapters/psych_adapter",
        "./adapters/psych_adapter",
        "adapters/psych_adapter",
        "/kaggle/working/ai-consultant-chatbot/adapters/psych_adapter",
        "/content/ai-consultant-chatbot/adapters/psych_adapter",
    ],
}

NGROK_TOKEN = get_my_secret("NGROK_AUTH_TOKEN")
if not NGROK_TOKEN:
    NGROK_TOKEN = "PASTE_YOUR_NGROK_TOKEN_HERE"

MAX_CONTEXT_TOKENS = 10000


def resolve_adapter_path(adapter_key):
    candidates = ADAPTER_PATH_CANDIDATES.get(adapter_key, [])
    for path in candidates:
        if os.path.isdir(path):
            return path

    checked = \"\n\".join(f"- {path}" for path in candidates)
    raise FileNotFoundError(
        f"Не удалось найти папку для адаптера '{adapter_key}'. "
        f"Проверь, что он распакован в один из путей:\n{checked}"
    )


In [6]:
print("Загрузка базовой модели...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

print("Загрузка токенизатора...")
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL_NAME,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

BUSINESS_ADAPTER_PATH = resolve_adapter_path("business")
LEGAL_ADAPTER_PATH = resolve_adapter_path("legal")
PSYCH_ADAPTER_PATH = resolve_adapter_path("psych")

print("Найденные пути адаптеров:")
print("business:", BUSINESS_ADAPTER_PATH)
print("legal:   ", LEGAL_ADAPTER_PATH)
print("psych:   ", PSYCH_ADAPTER_PATH)

print("Загрузка business adapter...")
model = PeftModel.from_pretrained(
    base_model,
    BUSINESS_ADAPTER_PATH,
    adapter_name="business"
)

print("Загрузка legal adapter...")
model.load_adapter(
    LEGAL_ADAPTER_PATH,
    adapter_name="legal"
)

print("Загрузка psych adapter...")
model.load_adapter(
    PSYCH_ADAPTER_PATH,
    adapter_name="psych"
)

model.eval()
print("Модель готова")


Загрузка базовой модели...


config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

Загрузка токенизатора...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/12.8M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Загрузка business adapter...
Загрузка legal adapter...
Создание hybrid adapter...
Модель готова


In [7]:
ADAPTER_DISPLAY_NAMES = {
    "business": "бизнес-консультант",
    "legal": "юридический консультант",
    "psych": "предпринимательский психолог",
}

SYSTEM_PROMPTS = {
    "business": """
Ты отвечаешь пользователю как опытный предприниматель с многолетним практическим опытом ведения бизнеса и управления компаниями.

Характер речи и тон:
- Чёткие и уверенные формулировки.
- Ориентация на практический результат: прибыль, эффективность, рост бизнеса.
- Объяснения простым и понятным языком без лишней теории.
- Использование деловой лексики: стратегия, масштабирование, юнит-экономика, рынок, конкурентные преимущества.

Структура ответов:
1. Краткий основной вывод или позиция.
2. Практическое объяснение, почему это работает или важно.
3. Конкретные действия или рекомендации (3–7 пунктов).
4. Краткое итоговое замечание или стратегический совет.

Запрещено: давать советы по незаконной деятельности, уходить в абстрактную теорию без практической пользы.
Если вопрос не напрямую связан с бизнесом — рассматривай его через призму карьеры, эффективности, управления ресурсами или предпринимательского мышления.
""",

    "legal": """
Ты отвечаешь пользователю как опытный юрист-практик с многолетним стажем юридической работы.

Характер речи:
- Чёткие, точные и понятные формулировки.
- Объяснение правовых принципов простым языком.
- Фокус на практических рекомендациях и реальных действиях.
- Уважительное и профессиональное общение.

Структура ответов:
1. Прямой ответ на вопрос.
2. Краткое объяснение правовых принципов или норм, которые применяются.
3. Практические шаги, которые стоит предпринять (3–7 пунктов).
4. Возможные риски и на что обратить внимание.

Запрещено: советовать незаконные действия, давать вводящие в заблуждение рекомендации или игнорировать возможные риски.
Если вопрос не юридический — рассматривай его через призму правовых рисков, защиты интересов и юридической грамотности.
""",

    "psych": """
Ты отвечаешь пользователю как предпринимательский психолог, который помогает владельцам бизнеса, руководителям и специалистам справляться со стрессом, выгоранием, тревогой, конфликтами и внутренними барьерами.

Характер речи:
- Спокойный, поддерживающий и уважительный тон.
- Эмпатия без сюсюканья и без ухода в пустую мотивацию.
- Практичные психологические рекомендации, которые можно применить в работе и жизни.
- Учет личных границ, устойчивости, эмоциональной нагрузки и когнитивных искажений.

Структура ответов:
1. Кратко назови суть проблемы или состояния.
2. Объясни, что может за этим стоять простым языком.
3. Дай 3–7 практических шагов самопомощи или коммуникации.
4. Отдельно отметь, когда лучше обратиться к профильному специалисту.

Запрещено: давать вредные советы, обесценивать состояние пользователя, изображать диагностику как окончательный медицинский вывод.
Если вопрос не психологический — рассматривай его через призму мотивации, устойчивости, командной динамики и ментальной нагрузки.
""",
}


def normalize_adapters(adapters, consultant=None):
    if isinstance(adapters, str):
        adapters = [adapters]

    if not isinstance(adapters, list):
        adapters = []

    cleaned = []
    seen = set()

    for adapter in adapters:
        key = str(adapter).strip().lower()
        if key in ADAPTER_DISPLAY_NAMES and key not in seen:
            cleaned.append(key)
            seen.add(key)

    if cleaned:
        return cleaned

    if consultant == "legal":
        return ["legal"]
    if consultant == "psych":
        return ["psych"]
    if consultant == "hybrid":
        return ["business", "legal"]

    return ["business"]


def build_system_prompt(adapters):
    adapters = normalize_adapters(adapters)

    if len(adapters) == 1:
        return SYSTEM_PROMPTS[adapters[0]]

    roles = ", ".join(ADAPTER_DISPLAY_NAMES[adapter] for adapter in adapters)

    prompt_parts = [
        f"""
Ты отвечаешь пользователю как единый экспертный AI-ассистент, который одновременно объединяет роли: {roles}.

Правила гибридного ответа:
- Дай один цельный, согласованный ответ.
- Не разрывай ответ на независимые роли, если это не помогает пониманию.
- Если между подходами есть компромисс или напряжение, обозначь это явно.
- В приоритете: польза, конкретика, ясность и безопасность рекомендаций.
- В финале дай собранный практический план действий.
""".strip()
    ]

    for adapter in adapters:
        prompt_parts.append(SYSTEM_PROMPTS[adapter].strip())

    return "\n\n".join(prompt_parts)


def activate_adapters(adapters):
    adapters = normalize_adapters(adapters)

    if len(adapters) == 1:
        model.set_adapter(adapters[0])
        return adapters[0]

    combo_name = "hybrid__" + "__".join(adapters)
    available_adapters = []

    if hasattr(model, "peft_config") and isinstance(model.peft_config, dict):
        available_adapters = list(model.peft_config.keys())

    if combo_name not in available_adapters:
        model.add_weighted_adapter(
            adapters=adapters,
            weights=[1.0 / len(adapters)] * len(adapters),
            adapter_name=combo_name,
            combination_type="linear",
        )

    model.set_adapter(combo_name)
    return combo_name


adapter_lock = threading.Lock()


In [8]:
app = Flask(__name__)


@app.route("/generate", methods=["POST"])
def generate():
    try:
        data = request.json
        if not data:
            return jsonify({"error": "Нет данных", "status": "error"}), 400

        user_message = data.get("message", "")
        history = data.get("history", [])
        consultant = data.get("consultant", "business")
        adapters = normalize_adapters(data.get("adapters"), consultant=consultant)

        if not user_message:
            return jsonify({"error": "Пустое сообщение", "status": "error"}), 400

        system_prompt = build_system_prompt(adapters)
        messages = [{"role": "system", "content": system_prompt}]

        system_tokens = len(tokenizer.encode(system_prompt))
        user_message_tokens = len(tokenizer.encode(user_message))
        current_tokens = system_tokens + user_message_tokens

        for msg in reversed(history):
            msg_tokens = len(tokenizer.encode(msg.get("content", "")))
            if current_tokens + msg_tokens < MAX_CONTEXT_TOKENS:
                messages.insert(1, {"role": msg["role"], "content": msg["content"]})
                current_tokens += msg_tokens
            else:
                break

        messages.append({"role": "user", "content": user_message})

        with adapter_lock:
            active_adapter_runtime = activate_adapters(adapters)

            text = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=False,
            )
            model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

            with torch.no_grad():
                generated_ids = model.generate(
                    **model_inputs,
                    max_new_tokens=800,
                    temperature=0.7,
                    top_p=0.9,
                    do_sample=True,
                    pad_token_id=tokenizer.eos_token_id,
                )

            generated_ids = [
                output[len(inp):]
                for inp, output in zip(model_inputs.input_ids, generated_ids)
            ]
            response_text = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

            del model_inputs, generated_ids

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

        return jsonify({
            "response": response_text,
            "active_adapters": adapters,
            "adapter_runtime": active_adapter_runtime,
            "status": "success"
        })

    except Exception as e:
        import traceback
        traceback.print_exc()
        return jsonify({"error": str(e), "status": "error"}), 500


@app.route("/health", methods=["GET"])
def health():
    return jsonify({
        "status": "ok",
        "model": BASE_MODEL_NAME,
        "adapters": ["business", "legal", "psych"],
        "consultants": ["business", "legal", "psych", "hybrid", "custom"],
        "hybrid_mode": "dynamic"
    })


@app.route("/", methods=["GET"])
def index():
    return jsonify({
        "message": "AI Consultant API is running",
        "endpoints": ["/generate", "/health"],
        "consultants": ["business", "legal", "psych", "hybrid", "custom"],
        "hybrid_mode": "dynamic"
    })


In [ ]:
if not NGROK_TOKEN or NGROK_TOKEN == "PASTE_YOUR_NGROK_TOKEN_HERE":
    raise ValueError("Укажи NGROK_AUTH_TOKEN в .env или вставь токен в переменную NGROK_TOKEN")

ngrok.kill()
ngrok.set_auth_token(NGROK_TOKEN)
public_url = ngrok.connect(5000)

print(f"\n{'='*50}")
print(f"URL:      {public_url}")
print(f"Endpoint: {public_url}/generate")
print(f"Health:   {public_url}/health")
print(f"{'='*50}\n")

app.run(host="0.0.0.0", port=5000, threaded=False)


                                                                                                    
URL:      NgrokTunnel: "https://cinnamoned-michelle-nonservile.ngrok-free.dev" -> "http://localhost:5000"
Endpoint: NgrokTunnel: "https://cinnamoned-michelle-nonservile.ngrok-free.dev" -> "http://localhost:5000"/generate
Health:   NgrokTunnel: "https://cinnamoned-michelle-nonservile.ngrok-free.dev" -> "http://localhost:5000"/health

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.19.2.2:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [11/Mar/2026 17:47:03] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/Mar/2026 17:48:59] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/Mar/2026 17:50:58] "POST /generate HTTP/1.1" 200 -
